# Redes Generativas Antagónicas (DCGAN) para Generación de Insectos


In [ ]:
import os
import random
import matplotlib.pyplot as plt
import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import torchvision.utils as vutils

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo de cómputo seleccionado: {device}")


In [ ]:
DATASET_DIR = r"c:\Users\lucia\Desktop\CNN dataset\dataset_v2"
TRAIN_DIR = os.path.join(DATASET_DIR, "train")

IMG_SIZE = 64
CHANNELS = 3
LATENT_DIM = 100
BATCH_SIZE = 64
LR = 0.0002
BETA1 = 0.5

print(f"Ruta de entrenamiento: {TRAIN_DIR}")
print(f"¿Existe la ruta?: {os.path.exists(TRAIN_DIR)}")

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

dataset = datasets.ImageFolder(root=TRAIN_DIR, transform=transform)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)

print(f"Dataset cargado con éxito: {len(dataset)} imágenes encontradas.")


In [ ]:
real_batch, _ = next(iter(dataloader))

def denormalize(tensor):
    return (tensor * 0.5) + 0.5

plt.figure(figsize=(8, 8))
plt.axis("off")
plt.title("Insectos Reales del Dataset (64x64 px)")
grid_img = vutils.make_grid(denormalize(real_batch[:16]), padding=2, normalize=False)
plt.imshow(np.transpose(grid_img.numpy(), (1, 2, 0)))
plt.show()


In [ ]:
class Generator(nn.Module):
    def __init__(self, latent_dim=100, ngf=64, channels=3):
        super(Generator, self).__init__()
        self.model = nn.Sequential(
            nn.ConvTranspose2d(latent_dim, ngf * 8, kernel_size=4, stride=1, padding=0, bias=False),
            nn.BatchNorm2d(ngf * 8),
            nn.ReLU(True),
            nn.ConvTranspose2d(ngf * 8, ngf * 4, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(ngf * 4),
            nn.ReLU(True),
            nn.ConvTranspose2d(ngf * 4, ngf * 2, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(ngf * 2),
            nn.ReLU(True),
            nn.ConvTranspose2d(ngf * 2, ngf, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(ngf),
            nn.ReLU(True),
            nn.ConvTranspose2d(ngf, channels, kernel_size=4, stride=2, padding=1, bias=False),
            nn.Tanh()
        )

    def forward(self, x):
        return self.model(x)

test_G = Generator(LATENT_DIM).to(device)
z_dummy = torch.randn(1, LATENT_DIM, 1, 1, device=device)
out_dummy = test_G(z_dummy)
print(f"Forma de salida del Generador: {out_dummy.shape}")


In [ ]:
class Discriminator(nn.Module):
    def __init__(self, channels=3, ndf=64):
        super(Discriminator, self).__init__()
        self.model = nn.Sequential(
            nn.Conv2d(channels, ndf, kernel_size=4, stride=2, padding=1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf, ndf * 2, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(ndf * 2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf * 2, ndf * 4, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(ndf * 4),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf * 4, ndf * 8, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(ndf * 8),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf * 8, 1, kernel_size=4, stride=1, padding=0, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.model(x)

test_D = Discriminator().to(device)
prob_dummy = test_D(out_dummy)
print(f"Forma de salida del Discriminador: {prob_dummy.shape}")


In [ ]:
def weights_init(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find('BatchNorm') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

netG = Generator(LATENT_DIM).to(device)
netD = Discriminator().to(device)

netG.apply(weights_init)
netD.apply(weights_init)

criterion = nn.BCELoss()
fixed_noise = torch.randn(16, LATENT_DIM, 1, 1, device=device)

optimizerG = optim.Adam(netG.parameters(), lr=LR, betas=(BETA1, 0.999))
optimizerD = optim.Adam(netD.parameters(), lr=LR, betas=(BETA1, 0.999))

print("Modelos instanciados e inicializados correctamente.")


In [ ]:
NUM_EPOCHS = 5

img_list = []
G_losses = []
D_losses = []

REAL_LABEL = 1.0
FAKE_LABEL = 0.0

print("Iniciando entrenamiento de la GAN...")
print("=" * 60)

for epoch in range(NUM_EPOCHS):
    for i, data in enumerate(dataloader):
        netD.zero_grad()
        real_cpu = data[0].to(device)
        b_size = real_cpu.size(0)
        label = torch.full((b_size,), REAL_LABEL, dtype=torch.float, device=device)
        
        output_real = netD(real_cpu).view(-1)
        errD_real = criterion(output_real, label)
        errD_real.backward()
        D_x = output_real.mean().item()
        
        noise = torch.randn(b_size, LATENT_DIM, 1, 1, device=device)
        fake = netG(noise)
        label.fill_(FAKE_LABEL)
        
        output_fake = netD(fake.detach()).view(-1)
        errD_fake = criterion(output_fake, label)
        errD_fake.backward()
        D_G_z1 = output_fake.mean().item()
        
        errD = errD_real + errD_fake
        optimizerD.step()
        
        netG.zero_grad()
        label.fill_(REAL_LABEL)
        
        output = netD(fake).view(-1)
        errG = criterion(output, label)
        errG.backward()
        D_G_z2 = output.mean().item()
        optimizerG.step()
        
        G_losses.append(errG.item())
        D_losses.append(errD.item())
        
        if i % 50 == 0:
            print(f"Época [{epoch+1}/{NUM_EPOCHS}] | Lote [{i}/{len(dataloader)}] "
                  f"| Loss_D: {errD.item():.4f} | Loss_G: {errG.item():.4f} "
                  f"| D(x): {D_x:.4f} | D(G(z)): {D_G_z1:.4f} / {D_G_z2:.4f}")

    with torch.no_grad():
        fake_samples = netG(fixed_noise).detach().cpu()
    img_list.append(vutils.make_grid(denormalize(fake_samples), padding=2, normalize=False))
    print(f"Época [{epoch+1}/{NUM_EPOCHS}] completada.")

print("=" * 60)
print("Entrenamiento finalizado!")


In [ ]:
plt.figure(figsize=(10, 5))
plt.title("Evolución de Pérdidas (Loss) durante el Entrenamiento")
plt.plot(G_losses, label="Generador (G)", alpha=0.7)
plt.plot(D_losses, label="Discriminador (D)", alpha=0.7)
plt.xlabel("Iteraciones")
plt.ylabel("Pérdida (Loss)")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
plt.figure(figsize=(12, 12))

if len(img_list) > 0:
    plt.subplot(1, 2, 1)
    plt.axis("off")
    plt.title("Insectos Generados por la GAN")
    plt.imshow(np.transpose(img_list[-1], (1, 2, 0)))

plt.subplot(1, 2, 2)
plt.axis("off")
plt.title("Insectos Reales del Dataset")
grid_real = vutils.make_grid(denormalize(real_batch[:16]), padding=2, normalize=False)
plt.imshow(np.transpose(grid_real.numpy(), (1, 2, 0)))

plt.show()


In [ ]:
def generar_insectos_nuevos(num_muestras=16):
    netG.eval()
    with torch.no_grad():
        ruido_nuevo = torch.randn(num_muestras, LATENT_DIM, 1, 1, device=device)
        insectos_sinteticos = netG(ruido_nuevo).cpu()
        cuadricula = vutils.make_grid(denormalize(insectos_sinteticos), nrow=4, padding=2)
        plt.figure(figsize=(8, 8))
        plt.axis("off")
        plt.title("Insectos Sintetizados por la GAN")
        plt.imshow(np.transpose(cuadricula.numpy(), (1, 2, 0)))
        plt.show()

generar_insectos_nuevos(num_muestras=16)
